# Alakazam B-agent — scaled self-play (recipe cloned from Archaludon A)

**Why Alakazam:** it's the ladder's **#1 deck (~32% of pilots)** and beats our Archaludon A
head-to-head (`rl_research/META_DISTRIBUTION_2026-06-30.md`). So Alakazam becomes the **B agent**.

**The recipe is already validated on Archaludon A.** That run (medium, 15.8M, same stable
orbit-wars recipe: LR 1e-4→1e-5, batch 256, 1 epoch, adaptive-ent target 0.15, KL 1.5) climbed
vs its held-out hand-coded rule agent 5% → 25% (it100) → 50% (it300) → **~70-75% (it875-925)** and
was still rising — a clean **GO**. We clone that recipe here unchanged; **no BC warm-start needed**.

**Deck-controlled, same as A:** Alakazam trainee pilots the rule agent's exact 60-card list;
`kaggle:alakazam` is held OUT of the training pool (`mixed_pool_heldout_alakazam.json`) and used
ONLY as the eval yardstick. Beating it >50% = self-play surpassed the matched heuristic.

**One addition vs A — the trained Archaludon A `best.pt` is in the league at a LOW weight (0.5).**
Via `--league-checkpoint`, it pilots its own Archaludon deck as a `model:archaludon_rl` opponent.
It's our strongest vendored agent (beats heuristic ~85%), so a low weight (~5% of games) gives
Alakazam a robustness/ceiling signal against it **without burying a weak early Alakazam** — the
other ~95% of games (self/past mirrors + weaker archetypes + heuristic/random) carry the reward
signal. Weight is **constant** (v1); a difficulty ramp is a deferred idea in
`rl_research/POOL_GATE_2026-06-30.md`.

**Read `eval: ... kaggle:alakazam`** every 25 iters — the honest yardstick:
- climbing through ~50% and still rising → Alakazam self-play works, promote/submit.
- flat in its start band → revisit (BC warm-start, more steps).

Note: no `--resume` yet — one long session (~12-16h at ~40s/iter). Checkpoints persist to Drive;
a dropped session loses optimizer/iter progress. Bump `ITERS` to fit budget.

In [ ]:
# Mount Drive (artifact persistence), then clone (or update) the repo and cd in.
import os

from google.colab import drive
drive.mount('/content/drive')

# Everything the runs produce is written under here so it survives the session and
# can be pulled back with scripts/download_artifacts.py. Keep this name in sync with
# DRIVE_BASE in that script ('ptcg_outputs').
DRIVE_OUTPUT = '/content/drive/MyDrive/ptcg_outputs'
for sub in ['', '/logs', '/checkpoints_sp_small', '/ablation_sp']:
    os.makedirs(DRIVE_OUTPUT + sub, exist_ok=True)

REPO_URL = "https://github.com/oshbocker/pokemon_tcg_ai_battle.git"
REPO_DIR = "/content/pokemon_tcg_ai_battle"
if not os.path.isdir(REPO_DIR):
    !git clone --depth 1 {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull --ff-only
%cd {REPO_DIR}

assert os.path.isfile("agent/cg/libcg.so"), (
    "agent/cg/libcg.so missing - push it to the repo (see .gitignore exception)."
)
import torch
print("repo:", os.getcwd(), "| Drive:", DRIVE_OUTPUT)
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())


In [ ]:
# Hardware topology - record this alongside the results.
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "(no GPU runtime selected)"
!nproc
!lscpu | grep -E "^CPU\(s\):|Thread\(s\) per core|Core\(s\) per socket|Model name"
!free -h | head -2


## Scaled long-run (`medium`, recipe cloned from Archaludon A)

Same validated recipe, retargeted to the Alakazam deck + `mixed_pool_heldout_alakazam.json`.
`--eval-opponents random,heuristic,kaggle:alakazam` runs the hand-coded Alakazam rule agent (the
held-out yardstick). Watch `ent` stays healthy and the `kaggle:alakazam` eval slope.

**Gate = `--gate-vs pool`, tuned for this run (2026-06-30):**
- **`--gate-games 40`** (not 120). The gate score is a *weighted mean* across ~9 opponents, so its
  sampling noise shrinks by ~√k_eff (k_eff≈7.2) — 40/opp already gives a tighter aggregate
  (±2.9%) than the old mirror gate while keeping the per-opponent breakdown legible (±8%).
  Derivation + recalibration recipe: `rl_research/GATE_GAMES_CALIBRATION.md`.
- **`--gate-every 50`** (not 25). On the Archaludon A run, real promotions were ≥75 iters apart
  after warmup, so 25 over-sampled 3–14×. 50 catches promotions with room to spare and halves the
  (serial, expensive) pool-gate cost. Eval stays at every 25 for slope resolution.

`best.pt` is crowned by the current net's weighted win-rate across the whole league — self + past
checkpoints + the manifest agents (Archaludon-rule/Starmie/Dragapult/romanrozen/heuristic/random)
+ the **trained `model:archaludon_rl` checkpoint** (asymmetric gate match on its own deck). Watch
the `gate pool NN% [self.. | archaludon.. | .. | archaludon_rl..]` breakdown — a much better
ladder proxy than a mirror number. Relative gate (beat the last-promoted score) with
`--gate-threshold 0.4` as a light absolute floor. `kaggle:alakazam` stays held OUT, so the eval
column is an honest yardstick the gate never optimises against.

In [ ]:
# -- Alakazam B-agent self-play long-run --
ITERS = 1200   # same budget as the validated Archaludon A run (~40s/iter); bump/cut to fit session
PROBE_OUT = f'{DRIVE_OUTPUT}/selfplay_alakazam_medium'
LOG = '/content/colab_selfplay_alakazam.txt'

# Trained Archaludon A best.pt — our strongest vendored agent — enters the league at a LOW weight
# (0.5, ~5% of games) so it robustifies Alakazam without burying it early. It pilots its OWN deck
# (asymmetric), labelled `archaludon_rl` to disambiguate from the `kaggle:archaludon` rule agent.
ARCH_CKPT = f'{DRIVE_OUTPUT}/probe_archaludon_medium_long/best.pt'
assert os.path.isfile(ARCH_CKPT), f"Archaludon A best.pt not found at {ARCH_CKPT} (run/sync the A notebook first)"

# PFSP (conservative first pass): after a 200-iter static warmup, re-weight the manifest
# opponents by the learner's live win-rate. `var` mode focuses games on contested (~50%)
# matchups. Modest bounds keep it gentle and never starve a pool agent: each opponent's
# weight stays within [0.5x, 2x] its manifest base (--pfsp-wmin 0.5 = the floor). self/past
# are untouched. Watch the new `| mix:` log line (realized share) + any `FORFEIT=` alarm.
!python scripts/train_selfplay.py \
    --deck agent/kaggle_agents/alakazam_deck.csv \
    --opp-manifest agent/opponents/mixed_pool_heldout_alakazam.json \
    --size medium --collector dist --workers 48 \
    --opponent self --league --pool-size 5 --snapshot-every 25 \
    --iters {ITERS} --games-per-iter 256 --epochs 1 --minibatch 256 \
    --lr 1e-4 --lr-final 1e-5 --ent-coef 0.04 --target-entropy 0.15 --target-kl 1.5 \
    --league-checkpoint {ARCH_CKPT}:0.5:agent/kaggle_agents/archaludon_deck.csv:archaludon_rl \
    --gate --gate-vs pool --gate-every 50 --gate-games 40 --gate-past 2 --gate-threshold 0.4 \
    --eval-every 25 --eval-games 120 \
    --eval-opponents random,heuristic,kaggle:alakazam \
    --pfsp --pfsp-mode var --pfsp-after 200 --pfsp-wmin 0.5 --pfsp-wmax 2.0 --pfsp-ema 0.8 \
    --device cuda --seed 0 \
    --out {PROBE_OUT} 2>&1 | tee {LOG}
!cp {LOG} {DRIVE_OUTPUT}/logs/   # persist log to Drive


## Results are on Drive — pull them to the laptop

Everything the runs produced lives under `MyDrive/ptcg_outputs/` (checkpoints, eval
CSVs, and the `logs/colab_*.txt` records). Nothing needs to be git-pushed from here.
The cell below just confirms what landed; back on the laptop, fetch it with rclone:

```bash
uv run python scripts/download_artifacts.py --logs                 # colab_*.txt -> rl_research/
uv run python scripts/download_artifacts.py --run ablation_sp      # A/B ckpts + CSVs -> outputs/
uv run python scripts/download_artifacts.py --run checkpoints_sp_small
```

Then paste the A/B table + RECOMMENDATION into `ABLATION_OPTION_RANK.md` and commit
the logs. (One-time rclone `gdrive` remote setup is in CLAUDE.md.)

In [ ]:
# Confirm what persisted to Drive.
!ls -lhR {DRIVE_OUTPUT} | sed -n '1,80p'
